# Week 3: Named Entity Extraction

This notebook shows my Week 3 entity extraction work.

## Deliverables

Regex extraction, taxonomy matching, 200 labeled remarks, evaluation, an 85% F1 target, and error analysis.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.entity_extractor import EntityExtractor

print("Project root:", project_root)

## Load the data

In [ ]:
cleaned_df = pd.read_csv(
    project_root / "data" / "processed" / "listing_sample_cleaned.csv"
)

labels_df = pd.read_csv(
    project_root / "data" / "processed" / "entity_labeling_template.csv"
)

print("Cleaned listings:", len(cleaned_df))
print("Reviewed labels:", len(labels_df))
print("Columns:", labels_df.columns.tolist())

## Deliverable 1: EntityExtractor with regex patterns

Regex finds bedrooms, bathrooms, price, and square footage. I included variations such as `bed`, `br`, `bd`, `bath`, and `ba`. I thought price would need phrases such as `priced at`, `cost`, and `listed at`, but the provided rule already searched for large numbers.

In [ ]:
extractor = EntityExtractor()

example = (
    "This 3 bedroom home has 2.5 bathrooms, 2000 square feet, "
    "and a community pool. The price is 850000."
)

print(extractor.extract_all(example))

## Try it on real listings

In [ ]:
for number, text in enumerate(
    cleaned_df["cleaned_remarks"].head(5),
    start=1,
):
    result = extractor.extract_all(text)
    print(f"\n--- LISTING {number} ---")
    print("Bedrooms:", result["bedrooms"])
    print("Bathrooms:", result["bathrooms"])
    print("Price:", result["price"])
    print("Square feet:", result["sqft"])
    print("Amenities found:", len(result["amenities"]))

## Deliverable 2: Amenity detection using the Week 1 taxonomy

Amenities are matched against the phrases in `taxonomy.json`.

## Deliverable 3: Labeled dataset with entity spans

This checks the 200 labeled remarks and their entity positions.

In [ ]:
span_file = (
    project_root / "data" / "processed" / "labeled_entities.jsonl"
)

records = []
label_counts = Counter()

with open(span_file, encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        records.append(record)

        for entity in record["entities"]:
            start = entity["start"]
            end = entity["end"]
            assert record["text"][start:end] == entity["text"]
            label_counts[entity["label"]] += 1

print("Records:", len(records))
print("Entity spans:", sum(label_counts.values()))
print("Labels:", dict(label_counts))

In [ ]:
print("Remark preview:")
print(records[0]["text"][:500])

print("\nEntity spans:")
for entity in records[0]["entities"][:10]:
    print(entity)

## Deliverable 4: Precision, recall, and F1 evaluation

Precision checks correctness, recall checks how much was found, and F1 balances both.

In [ ]:
metrics_file = project_root / "reports" / "week3_metrics.json"

with open(metrics_file, encoding="utf-8") as file:
    metrics = json.load(file)

metrics_table = pd.DataFrame(metrics).T
metrics_table[["precision", "recall", "f1"]] = (
    metrics_table[["precision", "recall", "f1"]] * 100
)

metrics_table

## Deliverable 5: Meet the 85% F1 target

Check whether the overall score is at least 85%.

In [ ]:
overall_f1 = metrics["overall"]["f1"]

print(f"Overall F1: {overall_f1:.2%}")
print("Meets 85% target:", overall_f1 >= 0.85)

## Deliverable 6: Error analysis

The main errors came from number words, unusual bathroom formats, and large numbers mistaken for prices. Square footage could also refer to a lot, deck, or garage instead of living space. Bathroom and price extraction need the most improvement.

## Conclusion

The extractor completed all Week 3 deliverables and passed the 85% overall F1 target.